In [1]:
library(lme4)
library(lmerTest)

Loading required package: Matrix


Attaching package: ‘lmerTest’


The following object is masked from ‘package:lme4’:

    lmer


The following object is masked from ‘package:stats’:

    step




In [2]:
df <- read.csv("concatenated_data/experiment3_acceptance_threshold.csv")
df <- df[(df$advanced=="True")|(df$bilingual=="False"),]
df$lang <- interaction(df$lang, df$bilingual)
df$lang <- as.character(df$lang)
df$lang[df$lang=="man.False"] <- "man"
df$lang[df$lang=="man.True"] <- "man_bilingual"
df$lang[df$lang=="min.True"] <- "min"

df[df$tone=="tone21",]$acceptance_threshold <- (df[df$tone=="tone21",]$acceptance_threshold-11)*-1 # flip the scale for tone21 for comparison
df$tone <- as.factor(df$tone)
contrasts(df$tone) <- c(-.5, .5, levels=c("tone21", "tone51"))
df$lang <- as.factor(df$lang)
print(contrasts(df$lang))
df$participant <- as.factor(df$participant)
df$bilingual <- as.factor(df$bilingual)
contrasts(df$bilingual) <- c(-.5, .5, levels=c("False", "True"))
df$strict <- as.factor(df$strict)
df$advanced <- as.factor(df$advanced)

              man_bilingual min
man                       0   0
man_bilingual             1   0
min                       0   1


In [3]:
# man_monolingual vs. man_bilingual vs. min vs. two tones
temp <- df
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

temp$tone <- factor(temp$tone, levels = c("tone21", "tone51"))
contrasts(temp$tone) <- c(-0.5, 0.5)
print(contrasts(temp$tone))
temp$lang <- as.factor(temp$lang)
print(contrasts(temp$lang))

model <- lmer(acceptance_threshold ~ lang*tone + (1|participant), data=temp)
summary(model)

       [,1]
tone21 -0.5
tone51  0.5
              man_bilingual min
man                       0   0
man_bilingual             1   0
min                       0   1


Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang * tone + (1 | participant)
   Data: temp

REML criterion at convergence: 167.9

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.5009 -0.4025 -0.1234  0.5585  1.5700 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.1332   0.3649  
 Residual                0.4493   0.6703  
Number of obs: 74, groups:  participant, 26

Fixed effects:
                        Estimate Std. Error       df t value Pr(>|t|)    
(Intercept)              5.13333    0.15445 38.78107  33.237  < 2e-16 ***
langman_bilingual       -0.08788    0.23745 38.78107  -0.370  0.71332    
langmin                 -0.31515    0.23745 38.78107  -1.327  0.19219    
tone1                    2.00000    0.24476 48.71630   8.171 1.09e-10 ***
langman_bilingual:tone1 -0.27273    0.37630 48.71630  -0.725  0.47206    
langmin:tone1            1.27273    0.37630 48

In [4]:
library(emmeans)

emm <- emmeans(
  model,
  ~ lang * tone,
  weights = "flat"
)

summary(emm)

tone_within_lang <- emmeans(
  model,
  ~ lang | tone,
  weights = "flat"
)

pairs(
  tone_within_lang
)

Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'



,lang,tone,emmean,SE,df,lower.CL,upper.CL
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,man,tone21,4.133333,0.1970541,61.24935,3.739332,4.527335
2,man_bilingual,tone21,4.181818,0.2301096,61.24935,3.721724,4.641913
3,min,tone21,3.181818,0.2301096,61.24935,2.721724,3.641913
4,man,tone51,6.133333,0.1970541,61.24935,5.739332,6.527335
5,man_bilingual,tone51,5.909091,0.2301096,61.24935,5.448996,6.369186
6,min,tone51,6.454545,0.2301096,61.24935,5.994451,6.914640


tone = tone21:
 contrast            estimate    SE   df t.ratio p.value
 man - man_bilingual  -0.0485 0.303 61.2  -0.160  0.9860
 man - min             0.9515 0.303 61.2   3.141  0.0072
 man_bilingual - min   1.0000 0.286 44.8   3.499  0.0030

tone = tone51:
 contrast            estimate    SE   df t.ratio p.value
 man - man_bilingual   0.2242 0.303 61.2   0.740  0.7406
 man - min            -0.3212 0.303 61.2  -1.060  0.5422
 man_bilingual - min  -0.5455 0.286 44.8  -1.908  0.1481

Degrees-of-freedom method: kenward-roger 
P value adjustment: tukey method for comparing a family of 3 estimates 

In [5]:
# man_monolingual vs. man_bilingual vs. min for tone21
temp <- df
temp <- temp[(temp$tone=="tone21"),]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)
model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 83.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.6690 -0.1778 -0.1584  0.8002  1.3354 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.1195   0.3457  
 Residual                0.4367   0.6608  
Number of obs: 37, groups:  participant, 26

Fixed effects:
                  Estimate Std. Error       df t value Pr(>|t|)    
(Intercept)        4.13333    0.19256 33.33969  21.465   <2e-16 ***
langman_bilingual  0.04848    0.29605 33.33969   0.164   0.8709    
langmin           -0.95152    0.29605 33.33969  -3.214   0.0029 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
            (Intr) lngmn_
lngmn_blngl -0.650       
langmin     -0.650  0.547

In [6]:
# man_monolingual vs. man_bilingual vs. min for tone51
temp <- df
temp <- temp[(temp$tone=="tone51"),]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)
model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 76.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-1.21363 -0.47153 -0.08228  0.53481  1.03970 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.4546   0.6743  
 Residual                0.1303   0.3609  
Number of obs: 37, groups:  participant, 26

Fixed effects:
                  Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)         6.1333     0.1975 27.1747  31.060   <2e-16 ***
langman_bilingual  -0.2242     0.3036 27.1747  -0.739    0.466    
langmin             0.3212     0.3036 27.1747   1.058    0.299    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
            (Intr) lngmn_
lngmn_blngl -0.650       
langmin     -0.650  0.872

In [7]:
# man_bilingual vs. min for tone21
temp <- df
temp <- temp[(temp$tone=="tone21"),]
temp <- temp[temp$lang!="man",]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 52.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.4244 -0.1554 -0.1554  0.6993  1.2588 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.1636   0.4045  
 Residual                0.5000   0.7071  
Number of obs: 22, groups:  participant, 11

Fixed effects:
            Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)   4.1818     0.2456 18.8537  17.025 6.68e-13 ***
langmin      -1.0000     0.3015 10.0000  -3.317  0.00779 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr)
langmin -0.614

In [8]:
# man_bilingual vs. min for tone51
temp <- df
temp <- temp[(temp$tone=="tone51"),]
temp <- temp[temp$lang!="man",]
temp$lang <- droplevels(temp$lang)
temp$tone <- droplevels(temp$tone)

model <- lmer(acceptance_threshold ~ lang + (1|participant), data=temp)
summary(model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: acceptance_threshold ~ lang + (1 | participant)
   Data: temp

REML criterion at convergence: 45.2

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.0506 -0.5331 -0.1733  0.6962  0.9502 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.6455   0.8034  
 Residual                0.1364   0.3693  
Number of obs: 22, groups:  participant, 11

Fixed effects:
            Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)   5.9091     0.2666 11.8935  22.165 4.87e-11 ***
langmin       0.5455     0.1575 10.0000   3.464  0.00608 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr)
langmin -0.295